In [7]:
import pickle
import pandas as pd
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, OneHotEncoder

# Load model dari file pickle
with open('best_model.pkl', 'rb') as file:
    model = pickle.load(file)

# Load dari file yang sudah disimpan sebelumnya
with open('ordinal_encoder.pkl', 'rb') as f:
    ord_enc = pickle.load(f)

with open('onehot_encoder.pkl', 'rb') as f:
    onehot_encoder = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('feature_list.pkl', 'rb') as f:
    feature_list = pickle.load(f)


In [44]:
def preprocess_input(data_df, ord_enc, onehot_encoder, scaler, feature_list):
    # Format gender
    data_df['person_gender'] = data_df['person_gender'].str.lower().str.replace(' ', '')
    data_df['person_gender'] = data_df['person_gender'].replace({'male': 'Male', 'female': 'Female'})

    # Encode previous loan defaults
    data_df['previous_loan_defaults_on_file'] = data_df['previous_loan_defaults_on_file'].map({'Yes': 1, 'No': 0}).astype(int)

    # Ordinal encode education (pakai encoder yg sudah di-fit sebelumnya)
    data_df['person_education'] = ord_enc.transform(data_df[['person_education']])

    # One-hot encode (pakai encoder yg sudah di-fit sebelumnya)
    onehot_cols = ['person_gender', 'person_home_ownership', 'loan_intent']
    encoded = onehot_encoder.transform(data_df[onehot_cols])
    encoded_df = pd.DataFrame(encoded, columns=onehot_encoder.get_feature_names_out(onehot_cols), index=data_df.index)

    # Gabung ke dataframe
    data_df = data_df.drop(columns=onehot_cols)
    data_df = pd.concat([data_df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

    # Scaling numeric columns
    num_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']
    data_df[num_cols] = scaler.transform(data_df[num_cols])

    # Pastikan urutan kolom sesuai yang dilatih
    for col in feature_list:
        if col not in data_df.columns:
            print(f"Kolom {col} tidak ada di data input. Menambahkan kolom dengan nilai 0.")
            data_df[col] = 0  # tambahkan kolom yang tidak ada dengan nilai 0

    return data_df[feature_list]

In [49]:
def predict_loan_status(input_data):
    processed = preprocess_input(input_data, ord_enc, onehot_encoder, scaler, feature_list)
    prediction = model.predict(processed)

    return 'Approved' if prediction[0] == 1 else 'Rejected'


In [50]:
def predict_full_data(self):
    # Simpan salinan original
    df_input = self.df.copy()

    # Simpan target aktual
    y_actual = df_input['loan_status']
    df_features = df_input.drop(columns=['loan_status'])

    # Preprocess
    X_processed = preprocess_input(
        data_df=df_features,
        ord_enc=self.ord_encoder,
        onehot_encoder=self.onehot_encoder,
        scaler=self.scaler,
        feature_list=self.feature_list
    )

    # Predict
    y_pred = self.model.predict(X_processed)

    # Gabungkan hasil ke DataFrame asli
    df_result = df_input.copy()
    df_result['loan_status_predicted'] = y_pred

    return df_result


In [51]:
# Baca file original
df_raw = pd.read_csv("Dataset_A_loan.csv")

# Simpan y_test
y_actual = df_raw['loan_status']

# Pisahkan fitur
df_features = df_raw.drop(columns=['loan_status'])

In [52]:
X_processed = preprocess_input(df_features, ord_enc, onehot_encoder, scaler, feature_list)
y_pred = model.predict(X_processed)

In [53]:
df_result = df_raw.copy()
df_result['loan_status_predicted'] = y_pred
df_result['is_correct'] = df_result['loan_status'] == df_result['loan_status_predicted']

# Tampilkan 5 baris pertama
df_result.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,loan_status_predicted,is_correct
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1,1,True
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0,0,True
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1,1,True
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1,1,True
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1,1,True


In [57]:
sample_input = pd.DataFrame([{
    'person_age': 24.0,
    'person_gender': 'Female',
    'person_education': 'Master',
    'person_income': 75000.0,
    'person_emp_exp': 0,
    'person_home_ownership': 'RENT',
    'loan_amnt': 25000.0,
    'loan_intent': 'PERSONAL',
    'loan_int_rate': 15,
    'loan_percent_income': 0.3,
    'cb_person_cred_hist_length': 3.7,
    'credit_score': 600,
    'previous_loan_defaults_on_file': 'No'
}])

result = predict_loan_status(sample_input)
print("Prediction Result:", result)

Prediction Result: Approved


In [56]:
sample_input2 = pd.DataFrame([{
    'person_age': 22.0,
    'person_gender': 'female',
    'person_education': 'High School',
    'person_income': 11282.0,
    'person_emp_exp': 0,
    'person_home_ownership': 'OWN',
    'loan_amnt': 2000.0,
    'loan_intent': 'EDUCATION',
    'loan_int_rate': 10.25,
    'loan_percent_income': 0.03,
    'cb_person_cred_hist_length': 3.0,
    'credit_score': 524,
    'previous_loan_defaults_on_file': 'Yes'
}])

result2 = predict_loan_status(sample_input2)
print("Prediction Result:", result2)

Prediction Result: Rejected


In [64]:
import pandas as pd

# Load data
df_raw = pd.read_csv("Dataset_A_loan.csv").dropna()

# Hitung ulang loan_percent_income secara manual
df_raw['loan_percent_income_manual'] = (df_raw['loan_amnt'] / df_raw['person_income'])

# Bandingkan nilai aslinya vs hasil hitungan manual
df_raw['is_match'] = df_raw['loan_percent_income'] == df_raw['loan_percent_income_manual'].round(2)

# Cek apakah semua baris cocok
print("Semua baris cocok?" , df_raw['is_match'].all())

# Kalau tidak semua cocok, tampilkan yang beda
if not df_raw['is_match'].all():
    print("\nContoh data yang tidak cocok:")
    print(df_raw.loc[~df_raw['is_match'], ['loan_amnt', 'person_income', 'loan_percent_income', 'loan_percent_income_manual']].head())


Semua baris cocok? False

Contoh data yang tidak cocok:
       loan_amnt  person_income  loan_percent_income  \
37035     3984.0        61291.0                 0.06   
41298     2325.0        35765.0                 0.06   
43684     3344.0        21570.0                 0.15   

       loan_percent_income_manual  
37035                    0.065001  
41298                    0.065008  
43684                    0.155030  


In [65]:
# Hitung min dan max per kolom numerik
min_max = pd.DataFrame({
    'Minimum': df_raw.min(numeric_only=True),
    'Maximum': df_raw.max(numeric_only=True)
})

min_max

,Minimum,Maximum
person_age,20.0,144.0
person_income,8000.0,5556399.0
person_emp_exp,0,125
loan_amnt,500.0,35000.0
loan_int_rate,5.42,20.0
loan_percent_income,0.0,0.66
cb_person_cred_hist_length,2.0,30.0
credit_score,390,807
loan_status,0,1
loan_percent_income_manual,0.000658,0.664186
